In [1]:
import sys
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

In [2]:
sys.path.insert(0, "..")   # pour importer src/ depuis notebooks/
from src.features.url import url_features, URL_FEATURES, NOT_FROM_URL

train = pd.read_csv("../data/processed/train.csv")
y = (train["status"] == "phishing").astype(int)

# Les 52 colonnes calculables à partir de l'URL seule
URL_ONLY = [c for c in URL_FEATURES if c not in NOT_FROM_URL]

rf_url = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(train[URL_ONLY], y)

In [3]:
def test_url(url):
    x = pd.DataFrame([url_features(url)])[URL_ONLY]
    p = rf_url.predict_proba(x)[0, 1]
    print(f"probabilité de phishing : {p:.2f}   {url}")

In [4]:
test_url("https://www.google.com/search?q=phishing")
test_url("https://fr.wikipedia.org/wiki/Hame%C3%A7onnage")
test_url("http://paypal-secure-login.verify-account.xyz/signin/update.php?id=8a3f9c2e")
test_url("https://secure-appleid.com.account-verify.top/login")

probabilité de phishing : 0.06   https://www.google.com/search?q=phishing
probabilité de phishing : 0.15   https://fr.wikipedia.org/wiki/Hame%C3%A7onnage
probabilité de phishing : 0.80   http://paypal-secure-login.verify-account.xyz/signin/update.php?id=8a3f9c2e
probabilité de phishing : 0.89   https://secure-appleid.com.account-verify.top/login


In [5]:
import numpy as np
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import roc_curve, f1_score, roc_auc_score

In [6]:
cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)
proba_url = cross_val_predict(RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
                              train[URL_ONLY], y, groups=train["group"], cv=cv,
                              method="predict_proba")[:, 1]

In [7]:
print("F1 (seuil 0,5) :", round(f1_score(y, proba_url >= 0.5), 4),
      "| ROC-AUC :", round(roc_auc_score(y, proba_url), 4))

F1 (seuil 0,5) : 0.8764 | ROC-AUC : 0.9506


In [8]:
def precision_at_prevalence(recall, fpr, prevalence=0.01):
    return recall * prevalence / (recall * prevalence + fpr * (1 - prevalence))


In [9]:
fpr, tpr, thresholds = roc_curve(y, proba_url)
rows = []
for max_fpr in [0.001, 0.01, 0.05]:
    i = np.searchsorted(fpr, max_fpr, side="right") - 1
    rows.append({"FP max": f"{max_fpr:.1%}", "seuil": round(thresholds[i], 3),
                 "rappel": round(tpr[i], 3), "FP réel": f"{fpr[i]:.2%}",
                 "précision si 1 %": round(precision_at_prevalence(tpr[i], fpr[i]), 2)})
pd.DataFrame(rows)

,FP max,seuil,rappel,FP réel,précision si 1 %
0,0.1%,0.979,0.123,0.09%,0.59
1,1.0%,0.862,0.535,0.96%,0.36
2,5.0%,0.663,0.791,4.97%,0.14


In [10]:
import json
import joblib
from pathlib import Path

In [11]:
cols = list(train.columns)
URL_COLS = cols[cols.index("length_url") : cols.index("statistical_report") + 1]
CONTENT_COLS = cols[cols.index("nb_hyperlinks") : cols.index("domain_with_copyright") + 1]
FULL_COLS = [c for c in URL_COLS + CONTENT_COLS if c != "random_domain"]

rf_full = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(train[FULL_COLS], y)

In [12]:
out = Path("../models")
joblib.dump(rf_full, out / "rf_url_content.joblib", compress=3)
joblib.dump(rf_url, out / "rf_url.joblib", compress=3)

['../models/rf_url.joblib']

In [13]:
config = {
    "url_content": {"model": "rf_url_content.joblib", "columns": FULL_COLS, "review": 0.81, "high": 0.933},
    "url_only":    {"model": "rf_url.joblib",         "columns": URL_ONLY,  "review": 0.862, "high": 0.979},
    "trained_on": "data/processed/train.csv (9 003 URL)",
    "thresholds_from": "probabilités hors-pli, GroupKFold(5), budgets de faux positifs 1 % (review) et 0,1 % (high)",
}

In [14]:
(out / "thresholds.json").write_text(json.dumps(config, indent=2, ensure_ascii=False))
print(sorted(p.name for p in out.iterdir()))

['rf_url.joblib', 'rf_url_content.joblib', 'thresholds.json']
